<a href="https://colab.research.google.com/github/ingridevv/data-engineering-zoomcamp/blob/main/week_6_batch/homework_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 6: Batch
Processing 4M+ taxi trips with Spark.
* Set up PySpark and create Spark sessions
* Read and process Parquet files at scale
* Repartition data for optimal performance
* Analyze millions of taxi trips with DataFrames
* Use Spark UI for monitoring jobs












In [1]:
!pip install pyspark

In [2]:
import pyspark
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
        .appName('homework_batch') \
        .getOrCreate()

In [4]:
spark.version

'4.0.2'

## Prepare the Data
*Download the Yellow-2025-11: wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet*
- Read the November 2025 Yellow into a Spark Dataframe.
- Repartition the Dataframe to 4 partitions and save it to parquet.
- What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? Select the answer which most closely matches.

In [5]:
# Download the data into the disk
!wget wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-08 22:09:23--  http://wget/
Resolving wget (wget)... failed: Name or service not known.
wget: unable to resolve host address ‘wget’
--2026-03-08 22:09:23--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 54.230.248.73, 54.230.248.222, 54.230.248.205, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|54.230.248.73|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M  26.1MB/s    in 2.6s    

2026-03-08 22:09:26 (26.1 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]

FINISHED --2026-03-08 22:09:26--
Total wall clock time: 3.0s
Downloaded: 1 files, 68M in 2.6s (26.1 MB/s)


## Yellow November 2025: Average Size
In this section I apply the repartition method to split the data into four different parts, then write it into parquet files.

In [6]:
# Read the November 2025 Yellow into a Spark Dataframe.
# Repartition the Dataframe to 4 partitions and save it to parquet.
spark.read.parquet('yellow_tripdata_2025-11.parquet').repartition(4).write.parquet('data/yellow_tripdata_2025-11_repartitioned.parquet')

In [7]:
# Check the average size of the parquet files

!ls -lh data/yellow_tripdata_2025-11_repartitioned.parquet/

total 102M
-rw-r--r-- 1 root root 26M Mar  8 22:10 part-00000-c399776a-12e9-4df1-8e3c-7fcd751dba4c-c000.snappy.parquet
-rw-r--r-- 1 root root 26M Mar  8 22:10 part-00001-c399776a-12e9-4df1-8e3c-7fcd751dba4c-c000.snappy.parquet
-rw-r--r-- 1 root root 26M Mar  8 22:10 part-00002-c399776a-12e9-4df1-8e3c-7fcd751dba4c-c000.snappy.parquet
-rw-r--r-- 1 root root 26M Mar  8 22:10 part-00003-c399776a-12e9-4df1-8e3c-7fcd751dba4c-c000.snappy.parquet
-rw-r--r-- 1 root root   0 Mar  8 22:10 _SUCCESS


## Count Records

In [8]:
spark_df = spark.read.parquet('data/yellow_tripdata_2025-11_repartitioned.parquet')

In [9]:
total_records = spark_df.count()
print(f'Total records are: {total_records}')

Total records are: 4181444


In [10]:
spark_df.show(20)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2025-11-07 15:04:17|  2025-11-07 15:39:15|              1|          7.3|         1|                 N|         262|    

In [11]:
from pyspark.sql import functions as F

In [12]:
help(F.to_date)

Help on function to_date in module pyspark.sql.functions.builtin:

to_date(col: 'ColumnOrName', format: Optional[str] = None) -> pyspark.sql.column.Column
    Converts a :class:`~pyspark.sql.Column` into :class:`pyspark.sql.types.DateType`
    using the optionally specified format. Specify formats according to `datetime pattern`_.
    By default, it follows casting rules to :class:`pyspark.sql.types.DateType` if the format
    is omitted. Equivalent to ``col.cast("date")``.

    .. _datetime pattern: https://spark.apache.org/docs/latest/sql-ref-datetime-pattern.html

    .. versionadded:: 2.2.0

    .. versionchanged:: 3.4.0
        Supports Spark Connect.

    Parameters
    ----------
    col : :class:`~pyspark.sql.Column` or column name
        input column of values to convert.
    format: literal string, optional
        format to use to convert date values.

    Returns
    -------
    :class:`~pyspark.sql.Column`
        date value as :class:`pyspark.sql.types.DateType` type.

 

In [13]:
spark_df.select("tpep_pickup_datetime").show(5)

+--------------------+
|tpep_pickup_datetime|
+--------------------+
| 2025-11-07 15:04:17|
| 2025-11-12 07:48:41|
| 2025-11-01 15:24:44|
| 2025-11-11 06:14:41|
| 2025-11-03 17:34:26|
+--------------------+
only showing top 5 rows


In [14]:
# How many taxi trips were there on a specific date?
spark_df.withColumn('specific_date', F.to_date(F.col('tpep_pickup_datetime'))) \
        .filter(F.col("specific_date") == "2025-11-15") \
        .count()


162604

## Longest Trip

In [20]:
# Largest
# trip distance, pickup_datetime, dropoff_datetime

timestamp_df = spark_df.withColumn("duration_hours",
                                   (F.unix_timestamp(F.col("tpep_dropoff_datetime")) -
                                        F.unix_timestamp(F.col("tpep_pickup_datetime"))) / 3600
)


timestamp_df.select("tpep_pickup_datetime", "tpep_dropoff_datetime", "duration_hours") \
            .orderBy(F.col("duration_hours").desc()) \
            .show(5)

+--------------------+---------------------+-----------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|   duration_hours|
+--------------------+---------------------+-----------------+
| 2025-11-26 20:22:12|  2025-11-30 15:01:00|90.64666666666666|
| 2025-11-27 04:22:41|  2025-11-30 09:19:35|76.94833333333334|
| 2025-11-03 10:42:55|  2025-11-06 14:55:45|76.21388888888889|
| 2025-11-07 11:23:22|  2025-11-10 08:40:41|69.28861111111111|
| 2025-11-18 17:12:47|  2025-11-21 12:17:37|67.08055555555555|
+--------------------+---------------------+-----------------+
only showing top 5 rows


## Least frequent zones

In [21]:
# Download zones data
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-08 22:15:39--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 54.230.248.73, 54.230.248.64, 54.230.248.205, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|54.230.248.73|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-08 22:15:39 (172 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [23]:
df_zones = spark.read \
    .option("header", "true") \
    .csv("taxi_zone_lookup.csv")

In [24]:
df_zones.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [35]:
df_counts = spark_df.groupBy("PULocationID").count()

In [36]:
df_result = df_counts.join(df_zones, df_counts.PULocationID == df_zones.LocationID)
df_result.orderBy("count").show(5)


+------------+-----+----------+-------------+--------------------+------------+
|PULocationID|count|LocationID|      Borough|                Zone|service_zone|
+------------+-----+----------+-------------+--------------------+------------+
|          84|    1|        84|Staten Island|Eltingville/Annad...|   Boro Zone|
|         105|    1|       105|    Manhattan|Governor's Island...| Yellow Zone|
|           5|    1|         5|Staten Island|       Arden Heights|   Boro Zone|
|         187|    3|       187|Staten Island|       Port Richmond|   Boro Zone|
|         204|    4|       204|Staten Island|   Rossville/Woodrow|   Boro Zone|
+------------+-----+----------+-------------+--------------------+------------+
only showing top 5 rows
